<a href="https://colab.research.google.com/github/BertrandOscarSaputra/dpr-agentic-ai/blob/zeamain/notebooks/train_indobert_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Fine-Tuning IndoBERT Sentimen Analisis DPR RI (Google Colab T4 GPU)
## Proyek DPR Agentic AI 2024–2029

Notebook resmi bagi **SI 1 (Data & Model Engineer)** untuk melatih model klasifikasi sentimen 3 kelas (**Negatif: 0, Netral: 1, Positif: 2**) berbasis **IndoBERT Base** menggunakan GPU T4 gratis.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

### ⚡ Cell 1: Install Dependencies

In [3]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy

### 🚀 Cell 2: Training Pipeline IndoBERT

In [5]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("🚀 Device Terdeteksi:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (Pastikan Runtime -> T4 GPU aktif!)")

MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IndoBERTSentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {
        "accuracy": round(acc, 4),
        "f1_macro": round(f1, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4)
    }

# 1. Muat Dataset (Pastikan file train.csv dan val.csv sudah di-upload ke Colab)
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")

train_dataset = IndoBERTSentimentDataset(train_df["text"].tolist(), train_df["label"].tolist(), tokenizer)
val_dataset = IndoBERTSentimentDataset(val_df["text"].tolist(), val_df["label"].tolist(), tokenizer)

# 2. Inisialisasi Model IndoBERT 3 Kelas
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: "Negatif", 1: "Netral", 2: "Positif"},
    label2id={"Negatif": 0, "Netral": 1, "Positif": 2}
)

# 3. Konfigurasi Pelatihan T4 GPU
training_args = TrainingArguments(
    output_dir="./indobert_checkpoints",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=100,  # Diubah dari warmup_ratio ke warmup_steps
    weight_decay=0.01,
    eval_strategy="epoch",  # Diubah dari evaluation_strategy ke eval_strategy (versi transformers terbaru)
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=20,
    fp16=True,
)

# 4. Eksekusi Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("🔥 Memulai Pelatihan IndoBERT di GPU T4...")
trainer.train()

# 5. Simpan Model
SAVE_DIR = "./indobert_sentiment_final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Pelatihan Selesai! Model tersimpan di '{SAVE_DIR}'.")

🚀 Device Terdeteksi: Tesla T4


[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🔥 Memulai Pelatihan IndoBERT di GPU T4...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Pelatihan Selesai! Model tersimpan di './indobert_sentiment_final'.


### 📦 Cell 3: Kompres & Unduh Otomatis ke Laptop

In [6]:
!zip -r indobert_sentiment_final.zip ./indobert_sentiment_final
from google.colab import files
files.download("indobert_sentiment_final.zip")

  adding: indobert_sentiment_final/ (stored 0%)
  adding: indobert_sentiment_final/tokenizer_config.json (deflated 43%)
  adding: indobert_sentiment_final/tokenizer.json (deflated 71%)
  adding: indobert_sentiment_final/config.json (deflated 57%)
  adding: indobert_sentiment_final/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>